# 🔬 Lifelong Image Retrieval using DwoPP & Episodic Metric Learning (Kaggle 2-GPU)
Notebook này hướng dẫn chi tiết cách chạy huấn luyện tăng trưởng và đánh giá hệ thống truy xuất ảnh suốt đời (**Lifelong Image Retrieval**) cho cả **4 Tasks** tuần tự trên Kaggle.

### ⚙️ Thiết kế mô hình & hàm Loss mới (DwoPP):
1. **Retrieval Projection Head**: Một lớp chiếu Conv2D cục bộ được tích hợp vào Decoupled Head của YOLO-World để ánh xạ đặc trưng vùng về 256 chiều.
2. **Episodic Hard-mining Metric Loss ($L_{eps}$)**: Triplet Loss áp dụng Batch-Hard mining để tối ưu hóa khoảng cách Euclidean của các mẫu dương cục bộ.
3. **Distillation without Positive Pairs ($L_{DwoPP}$)**: Hàm loss chưng cất tri thức từ mô hình nhiệm vụ trước nhưng loại bỏ hoàn toàn lớp tích cực (positive class) khỏi phân phối xác suất nhằm bảo toàn không gian metric mà không bị quên lãng thảm họa.
4. **Text Projection Layer**: Ánh xạ class embeddings văn bản từ 512 chiều về 256 chiều để đối sánh trực tiếp với đặc trưng vùng ảnh.

### ⚙️ Hỗ trợ Checkpoint Pre-trained:
* Nếu bạn đã pre-train trước các checkpoint phát hiện đối tượng (Object Detection) gốc của mô hình, bạn có thể cấu hình để nạp thẳng các checkpoint đó tại mỗi Task để học căn chỉnh không gian metric một cách nhanh chóng.

### ⚠️ Yêu cầu trước khi chạy:
1. Hãy chắc chắn rằng bạn đã kích hoạt **GPU T4 x2** trong phần settings của Kaggle (*Accelerator -> GPU T4 x2*).
2. Bật kết nối internet cho notebook (*Internet on*).

## 🛠️ Bước 1: Clone Repository & Submodules

In [1]:
import os
repo_url = "https://github.com/nta2112/OW_OVD-An-custom.git"
working_dir = "/kaggle/working/OW_OVD"

if not os.path.exists(working_dir):
    print("-> Đang clone repository từ GitHub...")
    !git clone {repo_url} {working_dir}
else:
    print("-> Repository đã tồn tại. Đang tiến hành cập nhật (git pull)...")
    %cd {working_dir}
    !git pull

%cd {working_dir}

# Tải mmyolo vào thư mục third_party nếu chưa có
if not os.path.exists("third_party/mmyolo"):
    print("-> Đang tải submodule mmyolo...")
    !git clone https://github.com/open-mmlab/mmyolo.git third_party/mmyolo
else:
    print("-> Submodule mmyolo đã có sẵn.")

-> Đang clone repository từ GitHub...
Cloning into '/kaggle/working/OW_OVD'...


remote: Enumerating objects: 1297, done.
remote: Counting objects: 100% (337/337), done.


remote: Compressing objects: 100% (241/241), done.


remote: Total 1297 (delta 235), reused 191 (delta 95), pack-reused 960 (from 1)
Receiving objects: 100% (1297/1297), 2.46 MiB | 11.27 MiB/s, done.


Resolving deltas: 100% (876/876), done.


/kaggle/working/OW_OVD
-> Đang tải submodule mmyolo...


Cloning into 'third_party/mmyolo'...


remote: Enumerating objects: 4968, done.
remote: Counting objects: 100% (1341/1341), done.
remote: Compressing objects: 100% (294/294), done.


remote: Total 4968 (delta 1133), reused 1047 (delta 1047), pack-reused 3627 (from 1)
Receiving objects: 100% (4968/4968), 3.62 MiB | 18.63 MiB/s, done.


Resolving deltas: 100% (3216/3216), done.


## 📦 Bước 2: Cài đặt Dependencies & Vá lỗi MMCV

In [2]:
print("-> 1. Thiết lập phiên bản PyTorch & Torchvision...")
!pip install -q torch==2.4.0+cu121 torchvision==0.19.0+cu121 --extra-index-url https://download.pytorch.org/whl/cu121

print("\n-> 2. Cài đặt MMCV từ wheel index...")
!pip install -q mmcv -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.4/index.html

print("\n-> 3. Cài đặt các thư viện bổ trợ...")
!pip install -q matplotlib pycocotools terminaltables mmengine prettytable wcwidth open_clip_torch transformers

print("\n-> 4. Cài đặt MMDetection...")
!pip install -q "mmdet>=3.1.0" --no-deps

print("\n-> 5. Cài đặt MMYOLO từ source...")
!pip install -q --no-build-isolation --no-deps third_party/mmyolo

print("\n-> 6. Vá lỗi kiểm tra phiên bản MMCV vật lý trên đĩa cứng...")
import site
import os
import glob
import shutil

def patch_file(file_path):
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            content = f.read()
        new_content = content
        for old_ver in ["'2.1.0'", "'2.2.0'", '"2.1.0"', '"2.2.0"']:
            new_content = new_content.replace(f"mmcv_maximum_version = {old_ver}", "mmcv_maximum_version = '2.3.0'")
        if new_content != content:
            with open(file_path, 'w', encoding='utf-8') as f:
                f.write(new_content)
            print(f"  [Vá lỗi] Đã cập nhật file: {file_path}")

def clear_pycache(root_dir):
    if not os.path.exists(root_dir):
        return
    for root, dirs, files in os.walk(root_dir):
        for d in dirs:
            if d == "__pycache__":
                pycache_path = os.path.join(root, d)
                try:
                    shutil.rmtree(pycache_path)
                except Exception:
                    pass

site_dirs = site.getsitepackages()
for s_dir in site_dirs:
    for pkg in ["mmdet", "mmyolo"]:
        pkg_dir = os.path.join(s_dir, pkg)
        patch_file(os.path.join(pkg_dir, "__init__.py"))
        clear_pycache(pkg_dir)

for init_file in glob.glob("**/mmyolo/__init__.py", recursive=True):
    patch_file(init_file)
    clear_pycache(os.path.dirname(init_file))
for init_file in glob.glob("**/mmdet/__init__.py", recursive=True):
    patch_file(init_file)
    clear_pycache(os.path.dirname(init_file))

paths_to_glob = [
    "/opt/conda/lib/python*/site-packages/mmdet/__init__.py",
    "/opt/conda/lib/python*/site-packages/mmyolo/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmdet/__init__.py",
    "/usr/local/lib/python*/dist-packages/mmyolo/__init__.py"
]
for path_pattern in paths_to_glob:
    for init_file in glob.glob(path_pattern):
        patch_file(init_file)
        clear_pycache(os.path.dirname(init_file))

print("\n-> 7. Kiểm tra import tất cả các package...")
import torch
import mmcv

real_mmcv_version = mmcv.__version__
mmcv.__version__ = '2.0.1'

import mmdet
import mmyolo
mmcv.__version__ = real_mmcv_version

print(f"  - torch: {torch.__version__} (CUDA: {torch.cuda.is_available()})")
print(f"  - mmcv: {mmcv.__version__}")
print(f"  - mmdet: {mmdet.__version__}")
print(f"  - mmyolo: {mmyolo.__version__}")
print("====== Khởi tạo môi trường hoàn tất! ======")

-> 1. Thiết lập phiên bản PyTorch & Torchvision...


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/799.0 MB ? eta -:--:--

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/799.0 MB 44.4 MB/s eta 0:00:18

     ╸━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/799.0 MB 186.7 MB/s eta 0:00:05

     ━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.2/799.0 MB 236.7 MB/s eta 0:00:04

...

  Preparing metadata (setup.py) ... done



-> 6. Vá lỗi kiểm tra phiên bản MMCV vật lý trên đĩa cứng...
  [Vá lỗi] Đã cập nhật file: /usr/local/lib/python3.12/dist-packages/mmdet/__init__.py


  [Vá lỗi] Đã cập nhật file: /usr/local/lib/python3.12/dist-packages/mmyolo/__init__.py


  [Vá lỗi] Đã cập nhật file: third_party/mmyolo/mmyolo/__init__.py
  [Vá lỗi] Đã cập nhật file: third_party/mmyolo/build/lib/mmyolo/__init__.py

-> 7. Kiểm tra import tất cả các package...


  - torch: 2.4.0+cu121 (CUDA: True)
  - mmcv: 2.2.0
  - mmdet: 3.3.0
  - mmyolo: 0.6.0
====== Khởi tạo môi trường hoàn tất! ======


## 🗂️ Bước 3: Định vị Dataset & Sinh Đặc trưng nhãn bằng CLIP

In [3]:
import json
import torch
import numpy as np
import os
import glob
from transformers import AutoTokenizer, CLIPTextModelWithProjection

os.makedirs('pretrained_models', exist_ok=True)
os.makedirs('data/IP102', exist_ok=True)
os.makedirs('data/texts/IP102', exist_ok=True)

weights_path = '/kaggle/input/models/nta212/yolo-world/pytorch/default/1/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth'
if not os.path.exists(weights_path):
    print("-> Đang tải pretrained weights của YOLO-World...")
    # !wget -O {weights_path} https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth

dataset_root = None
for path in [
    '/kaggle/input/datasets/nta212/ip102-for-object-detection',
    '/kaggle/input/ip102-for-object-detection',
    'data/IP102',
    '.'
]:
    if os.path.exists(os.path.join(path, 'train.json')):
        dataset_root = path
        break
if dataset_root is None:
    paths = glob.glob('/kaggle/input/**/train.json', recursive=True)
    if paths:
        dataset_root = os.path.dirname(paths[0])

print(f"-> Thư mục Dataset IP102: {dataset_root}")
class_names = [str(i) for i in range(102)]

class_texts = [[name] for name in class_names]
with open('data/texts/IP102/class_texts.json', 'w') as f:
    json.dump(class_texts, f)

print("-> Đang sinh text embeddings bằng CLIP...")
model_name = '/kaggle/input/models/yujkaggle/openaiclip-vit-base-patch32/pytorch/default/1'
tokenizer = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
clip_model = CLIPTextModelWithProjection.from_pretrained(model_name, local_files_only=True)
clip_model.eval()

embeddings = []
with torch.no_grad():
    for name in class_names:
        inputs = tokenizer(name, padding=True, return_tensors="pt")
        outputs = clip_model(**inputs)
        embed = outputs.text_embeds[0].cpu().numpy()
        embed = embed / np.linalg.norm(embed)
        embeddings.append(embed)

np.save('data/IP102/ip102_gt_embeddings.npy', np.array(embeddings))

num_att = len(class_names) * 25
torch.save({
    'att_embedding': torch.zeros(num_att, 512),
    'att_text': [f"att_{i}" for i in range(num_att)]
}, 'data/IP102/task_att_1_embeddings.pth')

thrs = [0.55]
pos_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
neg_dist = [{att_i: torch.zeros(10000) for att_i in range(num_att)} for _ in thrs]
torch.save({
    'positive_distributions': pos_dist,
    'negative_distributions': neg_dist
}, 'data/IP102/mowod_distribution_sim1.pth')
print("====== Khởi tạo và sinh đặc trưng nhãn hoàn tất! ======")

-> Đang tải pretrained weights của YOLO-World...
--2026-08-16 04:58:58--  https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth
Resolving huggingface.co (huggingface.co)... 

3.171.171.65, 3.171.171.128, 3.171.171.104, ...
Connecting to huggingface.co (huggingface.co)|3.171.171.65|:443... connected.


HTTP request sent, awaiting response... 

302 Found
Location: https://us.gcp.cdn.hf.co/xet-bridge-us/65bb7a71626a4c209906adf5/09dafb73b0d19d270cf20f7eeac6a7861303a753332d5df9917772ba23e4a47d?X-Xet-Cas-Uid=public&response-content-disposition=inline%3B+filename*%3DUTF-8%27%27yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth%3B+filename%3D%22yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth%22%3B&user_id=public&Expires=1786859939&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5nY3AuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjViYjdhNzE2MjZhNGMyMDk5MDZhZGY1LzA5ZGFmYjczYjBkMTlkMjcwY2YyMGY3ZWVhYzZhNzg2MTMwM2E3NTMzMzJkNWRmOTkxNzc3MmJhMjNlNGE0N2RcXD9YLVhldC1DYXMtVWlkPXB1YmxpYyZyZXNwb25zZS1jb250ZW50LWRpc3Bvc2l0aW9uPSomdXNlcl9pZD1wdWJsaWMiLCJDb25kaXRpb24iOnsiRGF0ZUxlc3NUaGFuIjp7IkVwb2NoVGltZSI6MTc4Njg1OTkzOX19fV19&Signature=MEUCIA7V-vsj7qO7-3zIf--ip8zRTa8LoZ5hM9531wN2xzgOAiEA4XMttBeXWjUepz-h8Jr7wXZhm3iU1gQiX7iPlEc6LK8_&Key-Pair-Id=01KXEF4KZ1B6FV465MAWR4M21F [following]
--2026-08-16 04:58:59--  https://us.gcp.cdn.hf.co/xet-bridge-us

HTTP request sent, awaiting response... 200 OK
Length: 441511203 (421M) [application/octet-stream]
Saving to: ‘pretrained_models/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth’

          pretraine   0%[                    ]       0  --.-KB/s               

...

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

CLIPTextModelWithProjection LOAD REPORT from: openai/clip-vit-base-patch32
Key                                                            | Status     |  | 
---------------------------------------------------------------+------------+--+-
vision_model.encoder.layers.{0...11}.mlp.fc1.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.weight            | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.weight   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.out_proj.bias   | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.self_attn.v_proj.bias     | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.mlp.fc2.bias              | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm1.bias          | UNEXPECTED |  | 
vision_model.embeddings.patch_embedding.weight                 | UNEXPECTED |  | 
vision_model.encoder.layers.{0...11}.layer_norm2.bias          | UNEXPECTED |  | 
vision_model.encoder.la

====== Khởi tạo và sinh đặc trưng nhãn hoàn tất! ======


## ⚙️ Bước 4: Khai báo Checkpoint Phát hiện có sẵn (Pre-trained Detection Checkpoints)
Nếu bạn đã huấn luyện trước các checkpoint phát hiện đối tượng gốc và muốn dùng checkpoint đó làm khởi tạo để chỉ tập trung tối ưu không gian metric truy xuất:
* Hãy điền đường dẫn checkpoint vào biến `PRETRAINED_DET_CHECKPOINTS` bên dưới.
* Nếu không có, hãy giữ giá trị `None` để hệ thống tự động học nối tiếp từ đầu.

In [4]:
PRETRAINED_DET_CHECKPOINTS = {
    "task_1": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/best_coco_Current class AP50_epoch_5.pth",
    "task_2": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t2.pth",
    "task_3": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t3.pth",
    "task_4": "/kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t4.pth"
}

print("-> Đã khai báo cấu hình checkpoints ban đầu.")

-> Đã khai báo cấu hình checkpoints ban đầu.


## 🛠️ Hàm bổ trợ ghi đè cấu hình để tránh lỗi khoảng trắng trong CLI
Việc ghi đè trực tiếp `load_from` vào file cấu hình Python sẽ tránh hoàn toàn các lỗi parser của MMEngine đối với tên file chứa khoảng trắng.

In [5]:
def prepare_config_with_checkpoint(task_idx, init_checkpoint):
    config_path = f"NewRetrieval_02/ip102_t{task_idx}_retrieval.py"
    with open(config_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()
    
    # Lọc bỏ dòng load_from cũ nếu có
    new_lines = [line for line in lines if not line.strip().startswith('load_from')]
    
    # Thêm khai báo load_from trực tiếp vào cuối file
    if init_checkpoint is not None:
        new_lines.append(f'\nload_from = {repr(init_checkpoint)}\n')
        
    with open(config_path, 'w', encoding='utf-8') as f:
        f.writelines(new_lines)
    print(f"-> Cấu hình {config_path} đã được cập nhật load_from = {init_checkpoint}")

## 🚀 Bước 5: Huấn luyện & Đánh giá Nhiệm vụ 1 (Task 1 - 7 Lớp đầu)
Huấn luyện khớp không gian metric của 7 lớp đầu tiên.

In [6]:
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t1_retrieval.py"
checkpoint_save_dir = "work_dirs/ip102_t1_retrieval"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_1"]
if init_checkpoint is None:
    init_checkpoint = "/kaggle/input/models/nta212/yolo-world/pytorch/default/1/yolo_world_v2_l_obj365v1_goldg_pretrain-a82b1fe3.pth"

# Ghi checkpoint tĩnh trực tiếp vào file config
prepare_config_with_checkpoint(1, init_checkpoint)

print(f"-> Bắt đầu huấn luyện Task 1...")
os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29500",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch"
]
subprocess.run(cmd, check=True)

-> Cấu hình NewRetrieval_02/ip102_t1_retrieval.py đã được cập nhật load_from = /kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/best_coco_Current class AP50_epoch_5.pth
-> Bắt đầu huấn luyện Task 1...


W0816 04:59:25.807000 133651089597568 torch/distributed/run.py:779] 
W0816 04:59:25.807000 133651089597568 torch/distributed/run.py:779] *****************************************
W0816 04:59:25.807000 133651089597568 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0816 04:59:25.807000 133651089597568 torch/distributed/run.py:779] *****************************************


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


/usr/local/lib/python3.12/dist-packages/mmdet/models/backbones/trident_resnet.py:244: SyntaxWarning: invalid escape sequence '\ '
  \ stage3(b2) /
/usr/local/lib/python3.12/dist-packages/mmdet/models/backbones/trident_resnet.py:244: SyntaxWarning: invalid escape sequence '\ '
  \ stage3(b2) /
/usr/local/lib/python3.12/dist-packages/mmdet/models/dense_heads/free_anchor_retina_head.py:290: SyntaxWarning: invalid escape sequence '\i'
  :math:`FL((1 - P_{a_{j} \in A_{+}}) * (1 - P_{j}^{bg}))`.
/usr/local/lib/python3.12/dist-packages/mmdet/models/dense_heads/free_anchor_retina_head.py:290: SyntaxWarning: invalid escape sequence '\i'
  :math:`FL((1 - P_{a_{j} \in A_{+}}) * (1 - P_{j}^{bg}))`.


/usr/local/lib/python3.12/dist-packages/mmengine/utils/dl_utils/setup_env.py:56: UserWarning: Setting MKL_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/mmengine/utils/dl_utils/setup_env.py:56: UserWarning: Setting MKL_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
  warnings.warn(


...

08/16 05:05:23 - mmengine - INFO - 86 has 1 predictions.
08/16 05:05:24 - mmengine - INFO - 101 has 1 predictions.
08/16 05:05:24 - mmengine - INFO - unknown has 1 predictions.


08/16 05:05:24 - mmengine - INFO - Wilderness Impact: {0.1: {50: np.float64(0.7020905923344948)}, 0.2: {50: np.float64(0.6977835723598435)}, 0.3: {50: np.float64(0.6777595926532097)}, 0.4: {50: np.float64(0.664461383575435)}, 0.5: {50: np.float64(0.654937837203847)}, 0.6: {50: np.float64(0.6458720987170697)}, 0.7: {50: np.float64(0.6345024599269957)}, 0.8: {50: np.float64(0.6070046082949309)}, 0.9: {50: np.float64(0.5420717694891285)}}
08/16 05:05:24 - mmengine - INFO - avg_precision: {0.1: {50: 0}, 0.2: {50: 0}, 0.3: {50: 0}, 0.4: {50: 0}, 0.5: {50: 0}, 0.6: {50: 0}, 0.7: {50: 0}, 0.8: {50: 0}, 0.9: {50: 0}}
08/16 05:05:24 - mmengine - INFO - known: ['14', '15', '16', '18', '22', '23', '24']
08/16 05:05:24 - mmengine - INFO - Absolute OSE (total_num_unk_det_as_known): {50: np.float64(19110.0)}
08/16 05:05:24 - mmengine - INFO - total_num_unk 1677
08/16 05:05:24 - mmengine - INFO - ['14', '15', '16', '18', '22', '23', '24', '25', '26', '37', '38', '39', '45', '46', '47', '48', '49', '5

08/16 05:05:24 - mmengine - INFO - The best checkpoint with 4.8497 coco/Current class AP50 at 1 epoch is saved to best_coco_Current class AP50_epoch_1.pth.


[rank0]:[W816 05:06:07.384289884 ProcessGroupNCCL.cpp:1168] Warning: WARNING: process group has NOT been destroyed before we destruct ProcessGroupNCCL. On normal program exit, the application should call destroy_process_group to ensure that any pending NCCL operations have finished in this process. In rare cases this process can exit before this point and block the progress of another member of the process group. This constraint has always been present,  but this warning has only been added since PyTorch 2.4 (function operator())


CompletedProcess(args=['torchrun', '--nproc_per_node=2', '--master_port=29500', 'third_party/mmyolo/tools/train.py', 'NewRetrieval_02/ip102_t1_retrieval.py', '--launcher', 'pytorch'], returncode=0)

In [7]:
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 1...")
best_checkpoint = "work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
cmd = [
    "python", "-u",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t1_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "1",
    "--query-cache", "query_cache_t1.pkl",
    "--gallery-cache", "gallery_cache_t1.pkl",
    "--output-report", "retrieval_lifelong_report_t1.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

-> Đang thực hiện đánh giá suốt đời sau Task 1...


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  atts = torch.load(att_embeddings)
/usr/local/lib/pyth

-> Fully patched transformers check_torch_load_is_safe across namespaces
      LIFELONG IMAGE RETRIEVAL EVALUATION PIPELINE      
-> Reading annotations...
-> Found 2176 query images and 2713 gallery images.
-> Extracting Query embeddings...
Loads checkpoint by local backend from path: work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth
[OurHead] Dynamically resizing self.att_embeddings from torch.Size([2550, 512]) to torch.Size([175, 512]) to match checkpoint.
-> Loading CLIP model: openai/clip-vit-base-patch32


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 3225.37it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Matching Queries:  86%|████████▋ | 1879/2176 [00:02<00:00, 846.24it/s]

Matching Queries:  99%|█████████▊| 2144/2176 [00:02<00:00, 871.18it/s]

Matching Queries: 100%|██████████| 2176/2176 [00:02<00:00, 868.01it/s]


-> Saved ROC Curve plot to: roc_curve_task_1.png

======================================== EVALUATION SUMMARY Task 1 ========================================
Global mAP:       0.2138
Recall@1:         0.5550
Recall@5:         0.8030
Recall@10:        0.8875
OOD AUROC:        0.4649
OOD FPR@TPR95:    0.9489
----------------------------------------
Plasticity:       0.2856
Forgetting (mAP): 0.0000 (0.00%)
Overall Change:   0.2856

-> Saved lifelong markdown evaluation report to: retrieval_lifelong_report_t1.md


CompletedProcess(args=['python', 'NewRetrieval_02/evaluate_retrieval_lifelong.py', '--config', 'NewRetrieval_02/ip102_t1_retrieval.py', '--checkpoint', 'work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth', '--dataset-root', '/kaggle/input/datasets/nta212/ip102-for-object-detection', '--current-task', '1', '--query-cache', 'query_cache_t1.pkl', '--gallery-cache', 'gallery_cache_t1.pkl', '--output-report', 'retrieval_lifelong_report_t1.md', '--history-file', 'history_metrics.json'], returncode=0)

## 🚀 Bước 6: Huấn luyện & Đánh giá Nhiệm vụ 2 (Task 2 - Thêm 6 lớp mới là 13 Lớp)
Nạp checkpoint học được từ Task 1 (hoặc checkpoint pretrain của Task 2 nếu khai báo) để tiếp tục huấn luyện.

In [8]:
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t2_retrieval.py"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_2"]
if init_checkpoint is None:
    init_checkpoint = "work_dirs/ip102_t1_retrieval/best_coco_Current class AP50_epoch_1.pth"

prepare_config_with_checkpoint(2, init_checkpoint)

print(f"-> Bắt đầu huấn luyện Task 2...")
os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29501",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch"
]
subprocess.run(cmd, check=True)

-> Cấu hình NewRetrieval_02/ip102_t2_retrieval.py đã được cập nhật load_from = /kaggle/input/models/nta212/task-1-ow-ovd-25-class/pytorch/default/3/ip102_t2.pth
-> Bắt đầu huấn luyện Task 2...


W0816 05:15:36.941000 140194354685056 torch/distributed/run.py:779] 
W0816 05:15:36.941000 140194354685056 torch/distributed/run.py:779] *****************************************
W0816 05:15:36.941000 140194354685056 torch/distributed/run.py:779] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0816 05:15:36.941000 140194354685056 torch/distributed/run.py:779] *****************************************


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \
/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


/usr/local/lib/python3.12/dist-packages/mmengine/utils/dl_utils/setup_env.py:56: UserWarning: Setting MKL_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/mmengine/utils/dl_utils/setup_env.py:56: UserWarning: Setting MKL_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed.
  warnings.warn(


08/16 05:15:59 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.
08/16 05:15:59 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.


...

08/16 05:26:57 - mmengine - INFO - 70 has 1 predictions.
08/16 05:26:57 - mmengine - INFO - 86 has 1 predictions.
08/16 05:26:57 - mmengine - INFO - 101 has 1 predictions.
08/16 05:26:57 - mmengine - INFO - unknown has 1 predictions.


08/16 05:26:57 - mmengine - INFO - Wilderness Impact: {0.1: {50: np.float64(0.5232149824424502)}, 0.2: {50: np.float64(0.5276211950394589)}, 0.3: {50: np.float64(0.5348750466243938)}, 0.4: {50: np.float64(0.5312953555878084)}, 0.5: {50: np.float64(0.5261103336525005)}, 0.6: {50: np.float64(0.525768886234697)}, 0.7: {50: np.float64(0.5179205368866411)}, 0.8: {50: np.float64(0.4969734996678231)}, 0.9: {50: np.float64(0.45718220853925307)}}
08/16 05:26:57 - mmengine - INFO - avg_precision: {0.1: {50: 0}, 0.2: {50: 0}, 0.3: {50: 0}, 0.4: {50: 0}, 0.5: {50: 0}, 0.6: {50: 0}, 0.7: {50: 0}, 0.8: {50: 0}, 0.9: {50: 0}}
08/16 05:26:57 - mmengine - INFO - known: ['14', '15', '16', '18', '22', '23', '24', '25', '26', '37', '38', '39', '45']
08/16 05:26:57 - mmengine - INFO - Absolute OSE (total_num_unk_det_as_known): {50: np.float64(29402.0)}
08/16 05:26:57 - mmengine - INFO - total_num_unk 1358
08/16 05:26:57 - mmengine - INFO - ['14', '15', '16', '18', '22', '23', '24', '25', '26', '37', '38', 

08/16 05:26:58 - mmengine - INFO - The best checkpoint with 3.1412 coco/Current class AP50 at 1 epoch is saved to best_coco_Current class AP50_epoch_1.pth.


[rank0]:[W816 05:27:40.910964330 ProcessGroupNCCL.cpp:1168] Warning: WARNING: process group has NOT been destroyed before we destruct ProcessGroupNCCL. On normal program exit, the application should call destroy_process_group to ensure that any pending NCCL operations have finished in this process. In rare cases this process can exit before this point and block the progress of another member of the process group. This constraint has always been present,  but this warning has only been added since PyTorch 2.4 (function operator())


CompletedProcess(args=['torchrun', '--nproc_per_node=2', '--master_port=29501', 'third_party/mmyolo/tools/train.py', 'NewRetrieval_02/ip102_t2_retrieval.py', '--launcher', 'pytorch'], returncode=0)

In [9]:
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 2...")
best_checkpoint = "work_dirs/ip102_t2_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
cmd = [
    "python", "-u",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t2_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "2",
    "--query-cache", "query_cache_t2.pkl",
    "--gallery-cache", "gallery_cache_t2.pkl",
    "--output-report", "retrieval_lifelong_report_t2.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

-> Đang thực hiện đánh giá suốt đời sau Task 2...


/usr/local/lib/python3.12/dist-packages/mmengine/optim/optimizer/zero_optimizer.py:11: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import \


/kaggle/working/OW_OVD/yolo_world/models/dense_heads/our_head_new.py:357: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  atts = torch.load(att_embeddings)
/usr/local/lib/pyth

-> Fully patched transformers check_torch_load_is_safe across namespaces
      LIFELONG IMAGE RETRIEVAL EVALUATION PIPELINE      
-> Reading annotations...
-> Found 2176 query images and 2713 gallery images.
-> Extracting Query embeddings...
Loads checkpoint by local backend from path: work_dirs/ip102_t2_retrieval/best_coco_Current class AP50_epoch_1.pth
[OurHead] Dynamically resizing self.att_embeddings from torch.Size([2550, 512]) to torch.Size([325, 512]) to match checkpoint.
-> Loading CLIP model: openai/clip-vit-base-patch32


Loading weights: 100%|██████████| 398/398 [00:00<00:00, 3288.32it/s, Materializing param=visual_projection.weight]
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Query Extraction: 100%|█████████▉| 2168/2176 [03:44<00:00,  9.79it/s]

Query Extraction: 100%|█████████▉| 2172/2176 [03:45<00:00,  9.86it/s]

Query Extraction: 100%|█████████▉| 2174/2176 [03:45<00:00,  9.86it/s]

Query Extraction: 100%|██████████| 2176/2176 [03:45<00:00,  9.64it/s]


-> Extracting Gallery embeddings...
Loads checkpoint by local backend from path: work_dirs/ip102_t2_retrieval/best_coco_Current class AP50_epoch_1.pth
[OurHead] Dynamically resizing self.att_embeddings from torch.Size([2550, 512]) to torch.Size([325, 512]) to match checkpoint.
-> Loading CLIP model: openai/clip-vit-base-patch32


## 🚀 Bước 7: Huấn luyện & Đánh giá Nhiệm vụ 3 (Task 3 - Thêm 6 lớp mới là 19 Lớp)

In [ ]:
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t3_retrieval.py"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_3"]
if init_checkpoint is None:
    init_checkpoint = "work_dirs/ip102_t2_retrieval/best_coco_Current class AP50_epoch_1.pth"

prepare_config_with_checkpoint(3, init_checkpoint)

print(f"-> Bắt đầu huấn luyện Task 3...")
os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29502",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch"
]
subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 3...")
best_checkpoint = "work_dirs/ip102_t3_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
cmd = [
    "python", "-u",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t3_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "3",
    "--query-cache", "query_cache_t3.pkl",
    "--gallery-cache", "gallery_cache_t3.pkl",
    "--output-report", "retrieval_lifelong_report_t3.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

## 🚀 Bước 8: Huấn luyện & Đánh giá Nhiệm vụ 4 (Task 4 - Thêm 5 lớp cuối là 25 Lớp)

In [ ]:
import subprocess
import os

config_path = "NewRetrieval_02/ip102_t4_retrieval.py"

init_checkpoint = PRETRAINED_DET_CHECKPOINTS["task_4"]
if init_checkpoint is None:
    init_checkpoint = "work_dirs/ip102_t3_retrieval/best_coco_Current class AP50_epoch_1.pth"

prepare_config_with_checkpoint(4, init_checkpoint)

print(f"-> Bắt đầu huấn luyện Task 4...")
os.environ["PYTHONPATH"] = "."
cmd = [
    "torchrun",
    "--nproc_per_node=2",
    "--master_port=29503",
    "third_party/mmyolo/tools/train.py",
    config_path,
    "--launcher", "pytorch"
]
subprocess.run(cmd, check=True)

In [ ]:
import subprocess
import os

print("-> Đang thực hiện đánh giá suốt đời sau Task 4...")
best_checkpoint = "work_dirs/ip102_t4_retrieval/best_coco_Current class AP50_epoch_1.pth"

os.environ["PYTHONPATH"] = "."
cmd = [
    "python", "-u",
    "NewRetrieval_02/evaluate_retrieval_lifelong.py",
    "--config", "NewRetrieval_02/ip102_t4_retrieval.py",
    "--checkpoint", best_checkpoint,
    "--dataset-root", dataset_root,
    "--current-task", "4",
    "--query-cache", "query_cache_t4.pkl",
    "--gallery-cache", "gallery_cache_t4.pkl",
    "--output-report", "retrieval_lifelong_report_t4.md",
    "--history-file", "history_metrics.json"
]
subprocess.run(cmd, check=True)

## 📊 Bước 9: Tổng hợp và hiển thị Ma trận học trọn đời (Lifelong Performance Matrix)
Hiển thị chi tiết bảng so sánh chất lượng truy xuất qua các pha huấn luyện để theo dõi mức độ ổn định của thuật toán chưng cất DwoPP.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Markdown

if os.path.exists("history_metrics.json"):
    with open("history_metrics.json", "r") as f:
        history = json.load(f)
        
    rows = []
    for stage, metrics in sorted(history.items()):
        rows.append({
            "Giai đoạn Đánh giá": stage.upper().replace("_", " "),
            "T1 mAP (7 lớp đầu)": f"{metrics.get('T1', {}).get('mAP', 0.0):.4f}",
            "T2 mAP (lớp 8-13)": f"{metrics.get('T2', {}).get('mAP', 0.0):.4f}",
            "T3 mAP (lớp 14-19)": f"{metrics.get('T3', {}).get('mAP', 0.0):.4f}",
            "T4 mAP (lớp 20-25)": f"{metrics.get('T4', {}).get('mAP', 0.0):.4f}",
        })
        
    df = pd.DataFrame(rows)
    display(Markdown("### 📈 Ma trận kết quả mAP học trọn đời:"))
    display(df)
    
    # Tính Forgetting & Plasticity cuối cùng sau Task 4
    if "task_4" in history and "task_1" in history:
        ap_t1_t1 = history["task_1"]["T1"]["mAP"]
        ap_t1_t4 = history["task_4"]["T1"]["mAP"]
        ap_t2_t2 = history["task_2"]["T2"]["mAP"]
        ap_t2_t4 = history["task_4"]["T2"]["mAP"]
        ap_t3_t3 = history["task_3"]["T3"]["mAP"]
        ap_t3_t4 = history["task_4"]["T3"]["mAP"]
        
        f1 = max(0.0, ap_t1_t1 - ap_t1_t4)
        f2 = max(0.0, ap_t2_t2 - ap_t2_t4)
        f3 = max(0.0, ap_t3_t3 - ap_t3_t4)
        forgetting = (f1 + f2 + f3) / 3.0
        plasticity = history["task_4"]["T4"]["mAP"]
        overall = plasticity - forgetting
        
        summary_md = f"""
### 📊 Chỉ số học trọn đời tích hợp (sau Task 4):
*   **Plasticity (Khả năng tiếp thu mới):** `{plasticity:.4f}`
*   **Forgetting (Độ quên lãng trung bình):** `{forgetting:.4f} ({forgetting*100:.2f}%)`
*   **Overall Change (Độ ổn định hệ thống):** `{overall:.4f}`
"""
        display(Markdown(summary_md))
else:
    print("-> File history_metrics.json không tồn tại. Hãy chạy đầy đủ các tác vụ huấn luyện và đánh giá trước.")

In [ ]:
# === Cell 25 ===
import zipfile
import glob
import os

zip_name = "/kaggle/working/retrieval_caches.zip"
pkl_files = glob.glob("*.pkl")
report_files = glob.glob("*.md")
png_files = glob.glob("*.png")
json_files = glob.glob("*.json")

files_to_zip = pkl_files + report_files + png_files + json_files
if files_to_zip:
    with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
        for f in files_to_zip:
            if os.path.exists(f) and f != "log.txt":
                zipf.write(f, os.path.basename(f))
    print(f"-> Đã đóng gói thành công {len(files_to_zip)} file vào {zip_name}")
else:
    print("-> Không tìm thấy file cache hoặc báo cáo nào để đóng gói.")
